In [1]:
import pandas as pd

In [2]:
# Lettura dati
# Solo anagrafica dei prodotti
prodotti = pd.read_csv('../data/PriceList.csv', sep = ";")
prodotti = prodotti[["ProductId", "Description", "Lev1Description", "Lev2Description", "Lev3Description", "VatCode", "PcsCar", "SaleUnitId", "Brand", "CurrentSalePrice", "CurrentSellingPrice"]]
prodotti.columns = ["id_prodotto", "descrizione", "cat1", "cat2", "cat3", "iva", "pzxcar", "um", "brand", "costo", "prezzo_vendita"]
prodotti = prodotti[prodotti["brand"] == 0]
prodotti = prodotti.drop(columns = ["brand"])

In [3]:
# Creazione di un nuovo id prodotto per anonimizzare i dati usando l'hashing
import hashlib

salt = "retail_project_2026"

prodotti["id_new"] = prodotti["id_prodotto"].astype(str).apply(
    lambda x: "COD_" + hashlib.sha256((salt + x).encode()).hexdigest()[:10].upper()
)

In [4]:
# Variazione dei prezzi per confondere ancora di più i prodotti usati
import numpy as np

# possibili variazioni
variazioni = np.array([-0.8, -0.7, -0.6, -0.5, -0.4, -0.3, -0.2,
                        0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8])

# generazione variazione casuale per ogni prodotto
prodotti["variazione"] = np.random.choice(variazioni, size=len(prodotti))

# applico la variazione
prodotti["costo"] = prodotti["costo"] + prodotti["variazione"]
prodotti["prezzo_vendita"] = prodotti["prezzo_vendita"] + prodotti["variazione"]

# arrotondo a 2 decimali
prodotti["costo"] = prodotti["costo"].round(2)
prodotti["prezzo_vendita"] = prodotti["prezzo_vendita"].round(2)

In [5]:
# Anonimizzazione anche dei nomi dei prodotti per evitare qualsiasi riferimento
# Rimozione dei brand e trasformazione in minuscolo
brands = [
# pasta
"barilla", "vivo barilla", "barilla bio", "barillabio",
"de cecco", "dececco",
"divella",
"voiello",
"garofalo",
"rumo",
"la molisana",

# biscotti / dolci
"mulino bianco", "mulinobianco",
"pavesi",
"balocco",
"colussi",
"gentilini",
"paneangeli",
"cameo",
"buitoni",
"motta",
"bauli",

# cioccolato
"ferrero",
"nutella",
"rocher",
"kinder",
"lindt",
"perugina",
"nestle", "nestlé",
"kitkat", "kit kat",
"milka",
"ritter",

# latte e latticini
"parmalat",
"granarolo",
"sterilgarda",
"lattebusche",
"vallelata",
"galbani",
"invernizzi",
"nonno nanni",
"valsoia",
"polenghi",

# conserve / tonno
"rio mare", "riomare",
"nostromo",
"callipo",
"cirio",
"mutti",
"star",
"knorr",
"heinz",

# bevande
"coca cola", "cocacola", "coca-cola",
"pepsi",
"fanta",
"sprite",
"red bull", "redbull",
"monster",
"schweppes",

# acqua
"san pellegrino", "sanpellegrino",
"levissima",
"ferrarelle",
"uliveto",
"lilia",
"santanna", "sant'anna",
"lauretana",

# tè / caffè
"lipton",
"twinings",
"pompadour",
"lavazza",
"illy",
"kimbo",
"segafredo",
"borbone",

# detersivi
"dash",
"ariel",
"lenor",
"vileda",
"ace",
"chanteclair",
"sole",
"ominobianco",
"bio presto",
"biopresto",

# carta / plastica
"scottex",
"tempo",
"fox",
"domopak",
"cuki",
"tenderly",

# surgelati
"findus",
"oro gel", "orogel",
"bofrost",
"4 salti", "quattro salti",

# snack
"pringles",
"lays", "lay's",
"san carlo", "sancarlo",
"amica chips",

# birra
"heineken",
"peroni",
"moretti",
"becks", "beck's",
"bud",
"budweiser",

# prodotti casa
"finish",
"calgon",
"cif",
"ajax",
"lysoform",
"vetril",

# igiene personale
"dove",
"nivea",
"loreal", "l'oréal",
"pantene",
"head shoulders", "head&shoulders",
"gillette",
"colgate",
"elmex",
"mentadent",

# macelleria
"aia",
"martini",
"amadori",

# ortofrutta
"melinda",
"fatina",
"ortoromi",
"naturella",

# safo
"maldera",
"senfter",

# bls
"starbucks", 
"montorsi",
"beretta", 
"mamma emma",
"zymil",
"pro drink",
"biraghi", 
"santa lucia",
"kefir",
"philadelphia",
"parmareggio",
"hopla",
"muller",
"plasmon"
]

import re

# normalizzazione descrizione
prodotti["descrizione_clean"] = prodotti["descrizione"].str.lower()

# mapping brand → codice anonimizzato
brand_map = {b: f"B{str(i+1).zfill(3)}" for i, b in enumerate(brands)}

# possibili abbreviazioni manuali
brand_alias = {
    "mulino bianco": ["mulino bianco", "mulinobianco", "m b", "mb"],
    "rio mare": ["rio mare", "riomare", "r mare"],
    "coca cola": ["coca cola", "cocacola", "coca-cola"],
    "san pellegrino": ["san pellegrino", "sanpellegrino"],
    "oro gel": ["oro gel", "orogel"],
    "san carlo": ["san carlo", "sancarlo"],
    "bio presto": ["bio presto", "biopresto"],
    "ominobianco": ["ominobianco", "omino bianco"],
    "aia": ["aia"],
    "martini": ["martini"],
    "melinda": ["melinda"],
    "fatina": ["fatina"],
    "ortoromi": ["ortoromi"],
    "naturella": ["naturella"],
    "maldera": ["maldera"],
    "senfter": ["senfter"],
    "amadori": ["amadori"],
    "starbucks": ["starbucks"],
    "valsoia": ["valsoia"],
    "montorsi": ["montorsi"],
    "beretta": ["beretta"],
    "mamma emma": ["mamma emma"],
    "zymil": ["zymil"],
    "biraghi": ["biraghi"],
    "santa lucia": ["s.lucia", "santa lucia", "s. lucia"],
    "pro drink": ["pro drink"],
    "kefir": ["kefir"],
    "philadelphia": ["philadelphia"],
    "parmareggio": ["parmareggio"],
    "hopla": ["hopla"],
    "muller": ["muller"],
    "plasmon": ["plasmon"]
}

# aggiungo alias mancanti direttamente alla lista
for brand in brands:
    if brand not in brand_alias:
        brand_alias[brand] = [brand]

# funzione per estrarre brand
def extract_brand(text):
    for brand, aliases in brand_alias.items():
        for alias in aliases:
            if re.search(rf"\b{re.escape(alias)}\b", text):
                return brand_map[brand]
    return None

# colonna brand anonimizzato
prodotti["brand_anon"] = prodotti["descrizione_clean"].apply(extract_brand)

# rimozione brand dalla descrizione
all_alias = [alias for aliases in brand_alias.values() for alias in aliases]
pattern = r'\b(' + '|'.join(map(re.escape, all_alias)) + r')\b'

prodotti["descrizione_clean"] = prodotti["descrizione_clean"].str.replace(pattern, "", regex=True)

# pulizia finale spazi
prodotti["descrizione_clean"] = prodotti["descrizione_clean"].str.replace(r"\s+", " ", regex=True).str.strip()

# flag prodotti con brand identificato
prodotti["brand_removed"] = prodotti["brand_anon"].notna()

In [6]:
# Estrazione di alcuni reparti in cui la maggior parte dei prodotti non ha un brand
macelleria = prodotti[(prodotti["cat1"] == "MACELLERIA") & (prodotti["brand_removed"] == False)]
ortofrutta = prodotti[(prodotti["cat1"] == "FRUTTA E VERDURA") & (prodotti["brand_removed"] == False)]
safo = prodotti[(prodotti["cat1"] == "SA.FO") & (prodotti["brand_removed"] == False)]
panetteria = prodotti[(prodotti["cat1"] == "PANETTERIA") & (prodotti["brand_removed"] == False)]

In [7]:
# Estrazione dei prodotti che possono far parte dell'anagrafica
anagrafica_base = prodotti[prodotti["brand_removed"] == True]

# Aggiunta di prodotti senza brand di default all'anagrafica
# Per ogni df estrapolato precedentemente, per i primi tre prendiamo solo la metà dei prodotti totali per renderli 
# omogenei agli altri gruppi
mace_sample = macelleria.sample(frac=0.05, random_state = 42) # random_state = 42 -> risultato riproducibile
ortofrutta_sample = ortofrutta.sample(frac = 0.3, random_state = 42)
safo_sample = safo.sample(frac= 0.3, random_state = 42)

# Aggiunta dei prodotti all'anagrafica
anagrafica = pd.concat([anagrafica_base, mace_sample, ortofrutta_sample, safo_sample, panetteria])

# Rimozione delle categorie di poco interesse
anagrafica = anagrafica[~anagrafica["cat1"].isin(["MANIFESTAZIONI", "MATERIALE DI CONSUMO"])]

In [8]:
# Selezione delle colonne necessarie
anagrafica_new = anagrafica[["id_new", "descrizione_clean", "cat1", "cat2", "cat3", "iva", "pzxcar", "um", "costo", "prezzo_vendita", "brand_anon" ]] \
    .rename(columns = {
        "id_new": "id_prodotto",
        "descrizione_clean": "descrizione",
        "brand_anon": "brand"
    })
    
# Rimozione dei prodotti della categoria ricorrenze prchè richiedono riordino periodico (evito complicazioni iniziali)
# Rimozione prodotti pasticceria
anagrafica_new = anagrafica_new[~anagrafica_new["cat1"].isin(["RICORRENZE", "PASTICCERIA"])]


In [9]:
# Rimozione dei prodotti con iva composta che vanno a complicare il lavoro
anagrafica_new = anagrafica_new[anagrafica_new["iva"] != "++"]

In [10]:
# Troncamento della descrizione del prodotto in modo da non avere nomi riconoscibili
anagrafica_new["descrizione"] = anagrafica_new["descrizione"].str[:15]

# Modifica di alcuni nomi di categorie
anagrafica_new["cat1"] = anagrafica_new["cat1"].replace("BLS", "FRESCHI")
anagrafica_new["cat1"] = anagrafica_new["cat1"].replace("SA.FO", "GASTRONOMIA")

In [14]:
anagrafica_new = anagrafica_new[anagrafica_new["prezzo_vendita"] > 0]
anagrafica_new = anagrafica_new[anagrafica_new["costo"] > 0]

In [15]:
# Scrittura dell'anagrafica pulita e anonimizzata 
anagrafica_new.to_csv("../data/products_master.csv")